In [5]:
follower_port = "/dev/tty.usbmodem5AAF2885151"
leader_port = "/dev/tty.usbmodem5AAF2885151"

follower_id = "follower_arm"
leader_id = "leader_arm"

# Teleoperate

In [ ]:
from lerobot.teleoperators.so101_leader import SO101LeaderConfig, SO101Leader
from lerobot.robots.so101_follower import SO101FollowerConfig, SO101Follower

robot_config = SO101FollowerConfig(
    port=follower_port,
    id=follower_id,
)

teleop_config = SO101LeaderConfig(
    port=leader_port,
    id=leader_id,
)

robot = SO101Leader(robot_config)
# teleop_device = SO101Leader(teleop_config)
robot.connect()
# teleop_device.connect()

while True:
    action = robot.get_action()
    robot.send_action(action)

Move all joints sequentially through their entire ranges of motion.
Recording positions. Press ENTER to stop...

-------------------------------------------
NAME            |    MIN |    POS |    MAX
shoulder_pan    |   2047 |   2047 |   2047
shoulder_lift   |   2047 |   2047 |   2047
elbow_flex      |   2047 |   2047 |   2047
wrist_flex      |   2047 |   2047 |   2047
wrist_roll      |   2047 |   2047 |   2047
gripper         |   2047 |   2047 |   2047

-------------------------------------------
NAME            |    MIN |    POS |    MAX
shoulder_pan    |   2047 |   2047 |   2047
shoulder_lift   |   2047 |   2047 |   2047
elbow_flex      |   2047 |   2047 |   2047
wrist_flex      |   2047 |   2047 |   2047
wrist_roll      |   2047 |   2047 |   2047
gripper         |   2047 |   2047 |   2047

-------------------------------------------
NAME            |    MIN |    POS |    MAX
shoulder_pan    |   2047 |   2047 |   2047
shoulder_lift   |   2047 |   2047 |   2047
elbow_flex      |   20

KeyboardInterrupt: 

In [ ]:
from lerobot.cameras.opencv.configuration_opencv import OpenCVCameraConfig
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.datasets.utils import hw_to_dataset_features
from lerobot.robots.so100_follower import SO100Follower, SO100FollowerConfig
from lerobot.teleoperators.so100_leader.config_so100_leader import SO100LeaderConfig
from lerobot.teleoperators.so100_leader.so100_leader import SO100Leader
from lerobot.utils.control_utils import init_keyboard_listener
from lerobot.utils.utils import log_say
from lerobot.utils.visualization_utils import _init_rerun
from lerobot.record import record_loop

NUM_EPISODES = 5
FPS = 30
EPISODE_TIME_SEC = 60
RESET_TIME_SEC = 10
TASK_DESCRIPTION = "My task description"

# Create the robot and teleoperator configurations
# camera_config = {"front": OpenCVCameraConfig(index_or_path=0, width=640, height=480, fps=FPS)}
robot_config = SO100FollowerConfig(
#     port=follower_port, id=follower_id, cameras=camera_confi
        port=follower_port, id=follower_id
)
teleop_config = SO100LeaderConfig(port=leader_port, id=leader_id)

# Initialize the robot and teleoperator
robot = SO100Follower(robot_config)
teleop = SO100Leader(teleop_config)

# Configure the dataset features
action_features = hw_to_dataset_features(robot.action_features, "action")
obs_features = hw_to_dataset_features(robot.observation_features, "observation")
dataset_features = {**action_features, **obs_features}

# Create the dataset
dataset = LeRobotDataset.create(
    repo_id="<hf_username>/<dataset_repo_id>",
    fps=FPS,
    features=dataset_features,
    robot_type=robot.name,
    use_videos=True,
    image_writer_threads=4,
)

# Initialize the keyboard listener and rerun visualization
_, events = init_keyboard_listener()
_init_rerun(session_name="recording")

# Connect the robot and teleoperator
robot.connect()
teleop.connect()

episode_idx = 0
while episode_idx < NUM_EPISODES and not events["stop_recording"]:
    log_say(f"Recording episode {episode_idx + 1} of {NUM_EPISODES}")

    record_loop(
        robot=robot,
        events=events,
        fps=FPS,
        teleop=teleop,
        dataset=dataset,
        control_time_s=EPISODE_TIME_SEC,
        single_task=TASK_DESCRIPTION,
        display_data=True,
    )

    # Reset the environment if not stopping or re-recording
    if not events["stop_recording"] and (episode_idx < NUM_EPISODES - 1 or events["rerecord_episode"]):
        log_say("Reset the environment")
        record_loop(
            robot=robot,
            events=events,
            fps=FPS,
            teleop=teleop,
            control_time_s=RESET_TIME_SEC,
            single_task=TASK_DESCRIPTION,
            display_data=True,
        )

    if events["rerecord_episode"]:
        log_say("Re-recording episode")
        events["rerecord_episode"] = False
        events["exit_early"] = False
        dataset.clear_episode_buffer()
        continue

    dataset.save_episode()
    episode_idx += 1

# Clean up
log_say("Stop recording")
robot.disconnect()
teleop.disconnect()
dataset.push_to_hub()

---


In [14]:
import sys
import os
# sys.path.insert(0, os.path.join(os.path.dirname(__file__), 'src'))

from lerobot.record import record
from lerobot.configs.policies import PreTrainedConfig
from lerobot.configs import parser

print("Starting Air Hockey with SO-101 robot...")

# Create record config
from lerobot.record import RecordConfig
from lerobot.robots.so101_follower.config_so101_follower import SO101FollowerConfig
from lerobot.cameras.opencv.configuration_opencv import OpenCVCameraConfig

# Robot config
robot_config = SO101FollowerConfig(
        port='/dev/tty.usbmodem5AAF2883661',
        id='follower_arm',
        cameras={
                'front': OpenCVCameraConfig(
                        index_or_path=1,
                        width=1920,
                        height=1080,
                        fps=30
                )
        }
)

# Dataset config
from lerobot.record import DatasetRecordConfig
dataset_config = DatasetRecordConfig(
        repo_id='npaka/eval_so101_final2',
        single_task='Air Hockey',
        fps=30,
        num_episodes=1,  # Start with 1 episode
        episode_time_s=60,
        video=True
)

# Policy config
policy_config = PreTrainedConfig.from_pretrained('outputs/migrated_air_hockey_5000')

# Record config
config = RecordConfig(
        robot=robot_config,
        dataset=dataset_config,
        policy=policy_config,
        display_data=True,
        play_sounds=True
)

try:
        print("Starting recording with policy control...")
        dataset = record(config)
        print(f"✅ Recording complete! Dataset saved to: {dataset.repo_id}")
except KeyboardInterrupt:
        print("🛑 Recording interrupted by user")
except Exception as e:
        print(f"❌ Error during recording: {e}")



INFO 2025-10-17 17:41:48 t/record.py:379 {'dataset': {'episode_time_s': 60,
             'fps': 30,
             'num_episodes': 1,
             'num_image_writer_processes': 0,
             'num_image_writer_threads_per_camera': 4,
             'private': False,
             'push_to_hub': True,
             'rename_map': {},
             'repo_id': 'npaka/eval_so101_final2',
             'reset_time_s': 60,
             'root': None,
             'single_task': 'Air Hockey',
             'tags': None,
             'video': True,
             'video_encoding_batch_size': 1},
 'display_data': True,
 'play_sounds': True,
 'policy': {'chunk_size': 100,
            'device': 'mps',
            'dim_feedforward': 3200,
            'dim_model': 512,
            'dropout': 0.1,
            'feedforward_activation': 'relu',
            'input_features': {'observation.images.front': {'shape': (3,
                                                                      1080,
                      

Starting Air Hockey with SO-101 robot...
Starting recording with policy control...


INFO 2025-10-17 17:41:50 a_opencv.py:179 OpenCVCamera(1) connected.
INFO 2025-10-17 17:41:50 follower.py:104 follower_arm SO101Follower connected.
INFO 2025-10-17 17:41:50 ls/utils.py:227 Recording episode 0
WARNING 2025-10-17 17:41:50 l/darwin.py:211 This process is not trusted! Input event monitoring will not be possible until it is added to accessibility clients.


❌ Error during recording: Failed to sync read 'Present_Position' on ids=[1, 2, 3, 4, 5, 6] after 1 tries. [TxRxResult] There is no status packet!


In [9]:
# Test script to check motor communication
from lerobot.robots.so101_follower import SO101Follower
from lerobot.robots.so101_follower.config_so101_follower import SO101FollowerConfig

config = SO101FollowerConfig(port='/dev/tty.usbmodem5AAF2885151', id='follower_arm')
robot = SO101Follower(config)

try:
    robot.bus.connect()
    print("Bus connected")
    
    # Try to read from individual motors
    for motor_id in [1, 2, 3, 4, 5, 6]:
        try:
            result = robot.bus.read("Present_Position", motor_id)
            print(f"Motor {motor_id}: {result}")
        except Exception as e:
            print(f"Motor {motor_id} error: {e}")
    
    robot.bus.disconnect()
except Exception as e:
    print(f"Bus connection error: {e}")


Bus connected
Motor 1 error: 1
Motor 2 error: 2
Motor 3 error: 3
Motor 4 error: 4
Motor 5 error: 5
Motor 6 error: 6


In [7]:
import cv2
import time

# Warm up iPhone camera
cap = cv2.VideoCapture(1)
if cap.isOpened():
    print("Camera opened successfully")
    for i in range(100):
        ret, frame = cap.read()
        if ret:
            print(f"Frame {i+1}: {frame.shape}")
        time.sleep(0.1)
    cap.release()
    print("Camera ready!")
else:
    print("Failed to open camera")


Camera opened successfully
Frame 1: (1080, 1920, 3)
Frame 2: (1080, 1920, 3)
Frame 3: (1080, 1920, 3)
Frame 4: (1080, 1920, 3)
Frame 5: (1080, 1920, 3)
Frame 6: (1080, 1920, 3)
Frame 7: (1080, 1920, 3)
Frame 8: (1080, 1920, 3)
Frame 9: (1080, 1920, 3)
Frame 10: (1080, 1920, 3)
Frame 11: (1080, 1920, 3)
Frame 12: (1080, 1920, 3)
Frame 13: (1080, 1920, 3)
Frame 14: (1080, 1920, 3)
Frame 15: (1080, 1920, 3)
Frame 16: (1080, 1920, 3)
Frame 17: (1080, 1920, 3)
Frame 18: (1080, 1920, 3)
Frame 19: (1080, 1920, 3)
Frame 20: (1080, 1920, 3)
Frame 21: (1080, 1920, 3)
Frame 22: (1080, 1920, 3)
Frame 23: (1080, 1920, 3)
Frame 24: (1080, 1920, 3)
Frame 25: (1080, 1920, 3)
Frame 26: (1080, 1920, 3)
Frame 27: (1080, 1920, 3)
Frame 28: (1080, 1920, 3)
Frame 29: (1080, 1920, 3)
Frame 30: (1080, 1920, 3)
Frame 31: (1080, 1920, 3)
Frame 32: (1080, 1920, 3)
Frame 33: (1080, 1920, 3)
Frame 34: (1080, 1920, 3)
Frame 35: (1080, 1920, 3)
Frame 36: (1080, 1920, 3)
Frame 37: (1080, 1920, 3)
Frame 38: (1080, 192

In [21]:
#!/usr/bin/env python3

"""
Final script to run air hockey with SO-101 robot using migrated policy
"""

import sys
import os
import subprocess

def run_air_hockey_command():
    """Run the air hockey recording command"""

    # First, clean up any existing dataset
    dataset_path = os.path.expanduser("~/.cache/huggingface/lerobot/npaka/eval_air_hockey_test")
    if os.path.exists(dataset_path):
        print(f"Removing existing dataset: {dataset_path}")
        import shutil
        shutil.rmtree(dataset_path)

    # Command to run
    cmd = [
        sys.executable, "-m", "lerobot.record",
        "--robot.type=so101_follower",
        "--robot.port=/dev/tty.usbmodem5AAF2883661",
        "--robot.id=follower_arm",
        "--robot.cameras={ front: {type: opencv, index_or_path: 1, width: 1280, height: 720, fps: 30}}",
        "--display_data=true",
        "--dataset.repo_id=npaka/eval_air_hockey_test",
        "--dataset.single_task=Air Hockey",
        "--dataset.num_episodes=1",
        "--dataset.episode_time_s=60",
        "--policy.device=cpu",
        "--policy.path=outputs/migrated_air_hockey"
    ]

    print("Running command:")
    print(" ".join(cmd))
    print("\nStarting air hockey recording...")

    try:
        # Run the command
        result = subprocess.run(cmd, cwd=os.getcwd())
        return result.returncode == 0
    except KeyboardInterrupt:
        print("🛑 Recording interrupted by user")
        return True
    except Exception as e:
        print(f"❌ Error running command: {e}")
        return False


success = run_air_hockey_command()
if success:
        print("\n🎉 Air hockey recording completed successfully!")
        print("Check the dataset at: https://huggingface.co/npaka/eval_air_hockey_test")
else:
        print("\n💥 Recording failed. Check the error messages above.")


Removing existing dataset: /Users/henry/.cache/huggingface/lerobot/npaka/eval_air_hockey_test
Running command:
/Users/henry/miniforge3/envs/lerobot/bin/python -m lerobot.record --robot.type=so101_follower --robot.port=/dev/tty.usbmodem5AAF2883661 --robot.id=follower_arm --robot.cameras={ front: {type: opencv, index_or_path: 1, width: 1280, height: 720, fps: 30}} --display_data=true --dataset.repo_id=npaka/eval_air_hockey_test --dataset.single_task=Air Hockey --dataset.num_episodes=1 --dataset.episode_time_s=60 --policy.device=cpu --policy.path=outputs/migrated_air_hockey

Starting air hockey recording...


INFO 2025-10-29 19:00:55 t/record.py:379 {'dataset': {'episode_time_s': 60,
             'fps': 30,
             'num_episodes': 1,
             'num_image_writer_processes': 0,
             'num_image_writer_threads_per_camera': 4,
             'private': False,
             'push_to_hub': True,
             'rename_map': {},
             'repo_id': 'npaka/eval_air_hockey_test',
             'reset_time_s': 60,
             'root': None,
             'single_task': 'Air Hockey',
             'tags': None,
             'video': True,
             'video_encoding_batch_size': 1},
 'display_data': True,
 'play_sounds': True,
 'policy': {'chunk_size': 100,
            'device': 'cpu',
            'dim_feedforward': 3200,
            'dim_model': 512,
            'dropout': 0.1,
            'feedforward_activation': 'relu',
            'input_features': {'observation.images.front': {'shape': (3,
                                                                      1080,
                   

Loading weights from local directory


WARNING 2025-10-29 19:00:56 a_opencv.py:237 OpenCVCamera(1) failed to set capture_width=1280 (actual_width=1920, width_success=True). Using actual width 1920.
WARNING 2025-10-29 19:00:56 a_opencv.py:244 OpenCVCamera(1) failed to set capture_height=720 (actual_height=1080, height_success=True). Using actual height 1080.
INFO 2025-10-29 19:00:57 a_opencv.py:180 OpenCVCamera(1) connected.
INFO 2025-10-29 19:00:57 follower.py:104 follower_arm SO101Follower connected.
INFO 2025-10-29 19:00:57 ls/utils.py:227 Recording episode 0
WARNING 2025-10-29 19:00:57 l/darwin.py:211 This process is not trusted! Input event monitoring will not be possible until it is added to accessibility clients.
Map: 100%|██████████| 1701/1701 [00:00<00:00, 5231.04 examples/s]
Generating train split: 1701 examples [00:00, 161571.46 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	Apple LLVM 15.0.0 (clang-1500.3.9.4)	 64 b


💥 Recording failed. Check the error messages above.


In [29]:
#!/usr/bin/env python3

"""
Final script to run air hockey with SO-101 robot using migrated 100K-step model
"""

import sys
import os
import subprocess

def run_air_hockey_command():
    """Run the air hockey recording command"""

    # First, clean up any existing dataset
    dataset_path = os.path.expanduser("~/.cache/huggingface/lerobot/npaka/eval_air_hockey_test")
    if os.path.exists(dataset_path):
        print(f"Removing existing dataset: {dataset_path}")
        import shutil
        shutil.rmtree(dataset_path)

    # Command to run - USING MIGRATED 100K MODEL
    cmd = [
        sys.executable, "-m", "lerobot.record",
        "--robot.type=so101_follower",
        "--robot.port=/dev/tty.usbmodem5AAF2883661",
        "--robot.id=follower_arm",
        "--robot.cameras={ front: {type: opencv, index_or_path: 1, width: 1280, height: 720, fps: 30}}",
        "--display_data=true",
        "--dataset.repo_id=npaka/eval_air_hockey_test",
        "--dataset.single_task=Air Hockey",
        "--dataset.num_episodes=1",
        "--dataset.episode_time_s=60",
        "--policy.device=cpu",
        "--policy.path=outputs/migrated_air_hockey_100k"  # <-- MIGRATED MODEL
    ]

    print("Running command:")
    print(" ".join(cmd))
    print("\n🎾 Starting air hockey with migrated 100K-step model...")

    try:
        # Run the command
        result = subprocess.run(cmd, cwd=os.getcwd())
        return result.returncode == 0
    except KeyboardInterrupt:
        print("🛑 Recording interrupted by user")
        return True
    except Exception as e:
        print(f"❌ Error running command: {e}")
        return False


success = run_air_hockey_command()
if success:
        print("\n🎉 Air hockey with migrated 100K model completed successfully!")
        print("🤖 Your robot should now play MUCH better air hockey!")
        print("Check the dataset at: https://huggingface.co/npaka/eval_air_hockey_test")
else:
        print("\n💥 Recording failed. Check the error messages above.")


Removing existing dataset: /Users/henry/.cache/huggingface/lerobot/npaka/eval_air_hockey_test
Running command:
/Users/henry/miniforge3/envs/lerobot/bin/python -m lerobot.record --robot.type=so101_follower --robot.port=/dev/tty.usbmodem5AAF2883661 --robot.id=follower_arm --robot.cameras={ front: {type: opencv, index_or_path: 1, width: 1280, height: 720, fps: 30}} --display_data=true --dataset.repo_id=npaka/eval_air_hockey_test --dataset.single_task=Air Hockey --dataset.num_episodes=1 --dataset.episode_time_s=60 --policy.device=cpu --policy.path=outputs/migrated_air_hockey_100k

🎾 Starting air hockey with migrated 100K-step model...


INFO 2025-10-29 19:36:02 t/record.py:379 {'dataset': {'episode_time_s': 60,
             'fps': 30,
             'num_episodes': 1,
             'num_image_writer_processes': 0,
             'num_image_writer_threads_per_camera': 4,
             'private': False,
             'push_to_hub': True,
             'rename_map': {},
             'repo_id': 'npaka/eval_air_hockey_test',
             'reset_time_s': 60,
             'root': None,
             'single_task': 'Air Hockey',
             'tags': None,
             'video': True,
             'video_encoding_batch_size': 1},
 'display_data': True,
 'play_sounds': True,
 'policy': {'chunk_size': 100,
            'device': 'cpu',
            'dim_feedforward': 3200,
            'dim_model': 512,
            'dropout': 0.1,
            'feedforward_activation': 'relu',
            'input_features': {'observation.images.front': {'shape': (3,
                                                                      1080,
                   

Loading weights from local directory


WARNING 2025-10-29 19:36:03 a_opencv.py:237 OpenCVCamera(1) failed to set capture_width=1280 (actual_width=1920, width_success=True). Using actual width 1920.
WARNING 2025-10-29 19:36:03 a_opencv.py:244 OpenCVCamera(1) failed to set capture_height=720 (actual_height=1080, height_success=True). Using actual height 1080.
INFO 2025-10-29 19:36:04 a_opencv.py:180 OpenCVCamera(1) connected.
INFO 2025-10-29 19:36:04 follower.py:104 follower_arm SO101Follower connected.
INFO 2025-10-29 19:36:04 ls/utils.py:227 Recording episode 0
WARNING 2025-10-29 19:36:04 l/darwin.py:211 This process is not trusted! Input event monitoring will not be possible until it is added to accessibility clients.
Traceback (most recent call last):
  File "/Users/henry/miniforge3/envs/lerobot/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/henry/miniforge3/envs/lerobot/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  F


💥 Recording failed. Check the error messages above.


In [ ]:
#!/usr/bin/env python3

import time
import torch
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.factory import make_policy

def test_inference_speed():
    print("Testing ACT policy inference speed...")
    
    # Load policy
    config = ACTConfig.from_pretrained('outputs/migrated_air_hockey_5000')
    policy = make_policy(config)
    policy.eval()
    
    # Create dummy input (camera image + robot state)
    batch = {
        'observation.images.front': torch.randn(1, 3, 1080, 1920),
        'observation.state': torch.randn(1, 6)
    }
    
    # Warm up
    with torch.no_grad():
        _ = policy(batch)
    
    # Time inference
    times = []
    for i in range(10):
        start = time.time()
        with torch.no_grad():
            action = policy(batch)
        end = time.time()
        times.append(end - start)
        print(".3f")
    
    avg_time = sum(times) / len(times)
    fps = 1.0 / avg_time
    print(".3f"
    print(".1f"
    
    # Test action prediction (100 steps)
    start = time.time()
    with torch.no_grad():
        action = policy(batch)  # This predicts 100 action steps
    end = time.time()
    
    print(".3f"
    print(".1f"


test_inference_speed()


In [ ]:
python -m lerobot.scripts.server.robot_client \
    --server_address=127.0.0.1:8080 \
    --robot.type=so101_follower \
    --robot.port=/dev/tty.usbmodem5AAF2883661 \
    --robot.id=follower_arm \
    --robot.cameras="{ front: {type: opencv, index_or_path: 1, width: 1920, height: 1080, fps: 30}}" \
    --task="Air Hockey" \
    --policy_type=act \
    --pretrained_name_or_path=AIBunCho/air-hockey-5000 \
    --policy_device=mps \
    --actions_per_chunk=10 \
    --chunk_size_threshold=0.7 \
    --aggregate_fn_name=weighted_average \
    --debug_visualize_queue_size=True

In [13]:
# Test camera independently
import cv2
cap = cv2.VideoCapture(1)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)

import time
for i in range(100):
    start = time.time()
    ret, frame = cap.read()
    print(f"Frame time: {(time.time() - start)*1000:.1f}ms")

Frame time: 16.7ms
Frame time: 14.0ms
Frame time: 43.3ms
Frame time: 46.6ms
Frame time: 37.4ms
Frame time: 39.4ms
Frame time: 41.7ms
Frame time: 42.5ms
Frame time: 41.2ms
Frame time: 48.4ms
Frame time: 40.9ms
Frame time: 35.7ms
Frame time: 42.4ms
Frame time: 46.2ms
Frame time: 41.1ms
Frame time: 36.7ms
Frame time: 41.4ms
Frame time: 42.1ms
Frame time: 40.8ms
Frame time: 47.5ms
Frame time: 46.1ms
Frame time: 31.1ms
Frame time: 45.7ms
Frame time: 48.8ms
Frame time: 35.5ms
Frame time: 38.7ms
Frame time: 43.5ms
Frame time: 37.4ms
Frame time: 41.8ms
Frame time: 48.2ms
Frame time: 40.9ms
Frame time: 47.2ms
Frame time: 31.2ms
Frame time: 46.9ms
Frame time: 40.7ms
Frame time: 39.3ms
Frame time: 38.0ms
Frame time: 44.2ms
Frame time: 41.4ms
Frame time: 46.4ms
Frame time: 38.9ms
Frame time: 37.6ms
Frame time: 42.7ms
Frame time: 44.9ms
Frame time: 41.4ms
Frame time: 38.2ms
Frame time: 39.6ms
Frame time: 44.4ms
Frame time: 40.7ms
Frame time: 47.1ms
Frame time: 40.3ms
Frame time: 36.9ms
Frame time: 

In [ ]:
 python -m lerobot.record --robot.type=so101_follower --robot.port=/dev/tty.usbmodem5AAF2883661 --robot.id=follower_arm --robot.cameras="{ front: {type: opencv, index_or_path: 1, width: 1280, height: 720, fps: 30}}" --task="Air Hockey" --num-episodes=1 --episode-time-s=10